In [ ]:
#IMPORTS
import pandas as pd
import numpy as np
import mysql.connector
import re
import unicodedata
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
#CONEXAO COM BD
SOURCE_HOST = "localhost"    
SOURCE_USER = "root"    
SOURCE_PASSWORD = "Nave31@03"

In [ ]:
#CAMINHOS DOS ARQUIVOS
# Caminhos relativos: funcionam em qualquer computador, desde que o notebook
# seja aberto a partir da pasta dados/ETL
CAMINHO_TACO  = "TACO.xlsx"
CAMINHO_IBGE  = "IBGE.xls"
# Tabela de Medidas Referidas para os Alimentos Consumidos no Brasil (POF 2008-2009)
# https://ftp.ibge.gov.br/Orcamentos_Familiares/Pesquisa_de_Orcamentos_Familiares_2008_2009/Tabela_de_Medidas_Referidas_para_os_Alimentos_Consumidos_no_Brasil/
CAMINHO_MEDIDAS = "tabelamedidas_bd.xls"
CAMINHO_SAIDA = "alimentos_tratados.csv"

In [ ]:
#IMPORT TABELA TACO
# header=None: a planilha tem o cabecalho em 2 linhas e repetido a cada pagina,
# entao lemos tudo cru e tratamos manualmente
df_taco_bruto = pd.read_excel(
    CAMINHO_TACO,
    sheet_name=0,
    header=None
)

print("Dimensão original:", df_taco_bruto.shape)

df_taco_bruto.head()

In [ ]:
#FIXANDO CABEÇALHO
cabecalho_taco   = [
    "numero_alimento",
    "descricao_alimentos",
    "umidade (%)",
    "energia (kcal)",
    "energia (kJ)",
    "proteina (g)",
    "lipideos (g)",
    "colesterol (mg)",
    "carboidrato (g)",
    "fibra_alimentar (g)",
    "cinzas (g)",
    "calcio (mg)",
    "magnesio (mg)",
    "numero_alimento_2",
    "manganes (mg)",
    "fosforo (mg)",
    "ferro (mg)",
    "sodio (mg)",
    "potassio (mg)",
    "cobre (mg)",
    "zinco (mg)",
    "retinol (mcg)",
    "re (mcg)",
    "rae (mcg)",
    "tiamina (mg)",
    "riboflavina (mg)",
    "piridoxina (mg)",
    "niacina (mg)",
    "vitamina_c (mg)"
]

In [ ]:
#SEPARANDO ALIMENTOS E GRUPOS
# Percorre a planilha linha a linha:
# - linha com numero na 1a coluna  -> alimento (recebe o grupo atual)
# - linha so com texto na 1a coluna -> nome de um novo grupo (ex: "Cereais e derivados")
# - linhas "Número do"/"Alimento"   -> cabecalho repetido da quebra de pagina, ignorar
# - "Legenda"                       -> fim dos dados
linhas_alimentos = []
grupos = []
grupo_atual = None

for _, linha in df_taco_bruto.iterrows():
    primeira = linha[0]

    if pd.notna(pd.to_numeric(primeira, errors="coerce")):
        linhas_alimentos.append(linha)
        grupos.append(grupo_atual)
        continue

    if not isinstance(primeira, str):
        continue

    texto = primeira.strip()
    if texto == "Legenda":
        break
    if texto.startswith("Número") or texto == "Alimento":
        continue
    if pd.isna(linha[1]):
        grupo_atual = texto

df_taco = pd.DataFrame(linhas_alimentos).reset_index(drop=True)
df_taco.columns = cabecalho_taco
df_taco.insert(0, "grupo", grupos)

print("Alimentos encontrados:", len(df_taco))
df_taco["grupo"].value_counts()

In [ ]:
#FUNÇÃO DE LIMPEZA DE VALORES
# Legenda da TACO / IBGE:
#   Tr  = traço (valor muito pequeno, arredondado)  -> nulo
#   *   = análise sendo reavaliada                  -> nulo
#   NA  = não se aplica                             -> nulo
#   -   = sem informação (IBGE)                     -> nulo
#   " " = análise não solicitada                    -> nulo
# Também troca vírgula decimal por ponto (ex: ",0,02" -> 0.02)
VALORES_NULOS = {"", "Tr", "*", "NA", "-"}

def limpar_valor(valor):
    if isinstance(valor, str):
        texto = valor.strip()
        if texto in VALORES_NULOS:
            return np.nan
        return pd.to_numeric(texto.lstrip(",").replace(",", "."), errors="coerce")
    return valor

In [ ]:
#LIMPEZA TACO
colunas_numericas_taco = cabecalho_taco[2:]

df_taco[colunas_numericas_taco] = (
    df_taco[colunas_numericas_taco]
    .map(limpar_valor)
    .astype(float)
)
df_taco["numero_alimento"] = df_taco["numero_alimento"].astype(int)
df_taco["descricao_alimentos"] = df_taco["descricao_alimentos"].str.strip()

# A coluna numero_alimento_2 é só a repetição do número (meio da página na planilha)
assert (df_taco["numero_alimento"] == df_taco["numero_alimento_2"]).all()
df_taco = df_taco.drop(columns="numero_alimento_2")

df_taco.head()

In [ ]:
# DF CEREAIS E DERIVADOS
df_cereais = df_taco[df_taco["grupo"] == "Cereais e derivados"]

print("Cereais:", len(df_cereais), "alimentos")
print(df_cereais.iloc[0][["numero_alimento", "descricao_alimentos"]].to_dict())
print(df_cereais.iloc[-1][["numero_alimento", "descricao_alimentos"]].to_dict())

In [ ]:
#PADRONIZANDO COLUNAS TACO
# Mesmos nomes da tabela "alimentos" do backend (FoodItensModel / data.sql)
COLUNAS_ALIMENTOS = [
    "nome", "umidade", "energia_kcal", "proteina", "lipideos", "colesterol",
    "carboidrato", "fibra", "cinzas", "calcio", "magnesio", "manganes",
    "fosforo", "ferro", "sodio", "potassio", "cobre", "zinco", "retinol",
    "tiamina", "riboflavina", "piridoxina", "niacina", "vitaminac"
]

mapa_taco = {
    "descricao_alimentos": "nome",
    "umidade (%)": "umidade",
    "energia (kcal)": "energia_kcal",
    "proteina (g)": "proteina",
    "lipideos (g)": "lipideos",
    "colesterol (mg)": "colesterol",
    "carboidrato (g)": "carboidrato",
    "fibra_alimentar (g)": "fibra",
    "cinzas (g)": "cinzas",
    "calcio (mg)": "calcio",
    "magnesio (mg)": "magnesio",
    "manganes (mg)": "manganes",
    "fosforo (mg)": "fosforo",
    "ferro (mg)": "ferro",
    "sodio (mg)": "sodio",
    "potassio (mg)": "potassio",
    "cobre (mg)": "cobre",
    "zinco (mg)": "zinco",
    "retinol (mcg)": "retinol",
    "tiamina (mg)": "tiamina",
    "riboflavina (mg)": "riboflavina",
    "piridoxina (mg)": "piridoxina",
    "niacina (mg)": "niacina",
    "vitamina_c (mg)": "vitaminac",
}

df_taco_final = df_taco.rename(columns=mapa_taco)
df_taco_final["fonte"] = "TACO"
df_taco_final["codigo_origem"] = df_taco_final["numero_alimento"]
df_taco_final = df_taco_final[["fonte", "codigo_origem", "grupo"] + COLUNAS_ALIMENTOS]

df_taco_final.head()

In [ ]:
#IMPORT TABELA IBGE
# header=3: as 3 primeiras linhas são o título da pesquisa (POF 2008-2009)
df_ibge = pd.read_excel(
    CAMINHO_IBGE,
    sheet_name=0,
    header=3
)
# Junta espaços repetidos: "CÓDIGO      DA PREPARAÇÃO" -> "CÓDIGO DA PREPARAÇÃO"
df_ibge.columns = [" ".join(str(c).split()) for c in df_ibge.columns]

# Remove o rodapé "Fonte: IBGE..." (linhas sem código numérico)
df_ibge = df_ibge[pd.to_numeric(df_ibge["CÓDIGO DO ALIMENTO"], errors="coerce").notna()].copy()
df_ibge["CÓDIGO DO ALIMENTO"] = df_ibge["CÓDIGO DO ALIMENTO"].astype(int)
df_ibge["CÓDIGO DA PREPARAÇÃO"] = df_ibge["CÓDIGO DA PREPARAÇÃO"].astype(int)

print("Dimensão IBGE:", df_ibge.shape)
df_ibge.head()

In [ ]:
#NOME DO ALIMENTO IBGE
# Mesmo padrão já usado no SQL: "Frango inteiro - Cozido(a)"
# Quando a preparação é "NAO SE APLICA", fica só a descrição
def montar_nome_ibge(linha):
    descricao = str(linha["DESCRIÇÃO DO ALIMENTO"]).strip().capitalize()
    preparacao = str(linha["DESCRIÇÃO DA PREPARAÇÃO"]).strip()
    if preparacao.upper() == "NAO SE APLICA":
        return descricao
    return f"{descricao} - {preparacao.capitalize()}"

df_ibge["nome"] = df_ibge.apply(montar_nome_ibge, axis=1)

df_ibge["nome"].head(10)

In [ ]:
#LIMPEZA E PADRONIZAÇÃO IBGE
# O IBGE não tem umidade nem cinzas -> ficam nulas
mapa_ibge = {
    "ENERGIA (kcal)": "energia_kcal",
    "PROTEÍNA (g)": "proteina",
    "LIPÍDEOS TOTAIS (g)": "lipideos",
    "COLESTEROL (mg)": "colesterol",
    "CARBOIDRATO (g)": "carboidrato",
    "FIBRA ALIMENTAR TOTAL (g)": "fibra",
    "CÁLCIO (mg)": "calcio",
    "MAGNÉSIO (mg)": "magnesio",
    "MANGANÊS (mg)": "manganes",
    "FÓSFORO (mg)": "fosforo",
    "FERRO (mg)": "ferro",
    "SÓDIO (mg)": "sodio",
    "POTÁSSIO (mg)": "potassio",
    "COBRE (mg)": "cobre",
    "ZINCO (mg)": "zinco",
    "RETINOL (mcg)": "retinol",
    "TIAMINA (VITAMINA B1) (mg)": "tiamina",
    "RIBOFLAVINA (VITAMINA B2) (mg)": "riboflavina",
    "PIRIDOXINA (VITAMINA B6) (mg)": "piridoxina",
    "NIACINA (VITAMINA B3) (mg)": "niacina",
    "VITAMINA C (mg)": "vitaminac",
}

faltando = set(mapa_ibge) - set(df_ibge.columns)
assert not faltando, f"Colunas não encontradas no IBGE: {faltando}"

df_ibge_final = df_ibge.rename(columns=mapa_ibge)
colunas_numericas_ibge = list(mapa_ibge.values())
df_ibge_final[colunas_numericas_ibge] = (
    df_ibge_final[colunas_numericas_ibge]
    .map(limpar_valor)
    .astype(float)
)

df_ibge_final["umidade"] = np.nan
df_ibge_final["cinzas"] = np.nan
df_ibge_final["fonte"] = "IBGE"
df_ibge_final["codigo_origem"] = df_ibge_final["CÓDIGO DO ALIMENTO"]
df_ibge_final["codigo_preparacao"] = df_ibge_final["CÓDIGO DA PREPARAÇÃO"]
df_ibge_final["grupo"] = None
df_ibge_final = df_ibge_final[["fonte", "codigo_origem", "codigo_preparacao", "grupo"] + COLUNAS_ALIMENTOS]

df_ibge_final.head()

In [ ]:
#IMPORT TABELA DE MEDIDAS CASEIRAS (IBGE)
# A tabela tem uma linha por alimento + preparação + tipo de medida, com o peso
# em gramas. Ficamos só com as medidas que o front oferece
# (frontend/src/components/MealModal.jsx). g e kg são conversões fixas
# (1 g e 1000 g) e L = 1000 ml, então não precisam de coluna.
MEDIDAS_IBGE = {
    103: "gramas_unidade",      # UNIDADE
    16: "gramas_colher_sopa",   # COLHER DE SOPA
    14: "gramas_colher_cha",    # COLHER DE CHA
    106: "gramas_xicara",       # XICARA DE CHA
    79: "gramas_ml",            # MILILITRO
}
COLUNAS_MEDIDAS = list(MEDIDAS_IBGE.values())

df_medidas_bruto = pd.read_excel(
    CAMINHO_MEDIDAS,
    sheet_name=0,
    header=3
)
df_medidas_bruto.columns = [" ".join(str(c).split()) for c in df_medidas_bruto.columns]
df_medidas_bruto = df_medidas_bruto[pd.to_numeric(df_medidas_bruto["CÓDIGO DO ALIMENTO"], errors="coerce").notna()]
print("Linhas na tabela de medidas:", len(df_medidas_bruto))

df_medidas = df_medidas_bruto[df_medidas_bruto["CÓDIGO DO TIPO DE MEDIDA"].isin(MEDIDAS_IBGE)].assign(
    codigo_origem=lambda d: d["CÓDIGO DO ALIMENTO"].astype(int),
    codigo_preparacao=lambda d: d["CÓDIGO DA PREPARAÇÃO"].astype(int),
    medida=lambda d: d["CÓDIGO DO TIPO DE MEDIDA"].map(MEDIDAS_IBGE),
)

# Pivot: transforma as linhas de medida em colunas (uma coluna por medida)
df_medidas = (
    df_medidas.pivot(index=["codigo_origem", "codigo_preparacao"], columns="medida", values="QUANTIDADE EM GRAMAS")
    .reindex(columns=COLUNAS_MEDIDAS)
    .astype(float)
    .reset_index()
)

print("Alimentos/preparações com alguma medida do front:", len(df_medidas))
df_medidas.head()

In [ ]:
#MEDIDAS NOS ALIMENTOS DA TACO
# A tabela de medidas só usa códigos do IBGE. Mas o IBGE.xls informa quando a
# composição de um alimento foi copiada da TACO (referência 2) e traz o nome
# exato do alimento na TACO. Por esse caminho levamos as medidas para a TACO.
# Se vários alimentos do IBGE apontam para o mesmo da TACO (ex: mandioca,
# aipim e macaxeira -> "Mandioca, cozida"), usamos a mediana dos gramas.
ligacao_taco = df_ibge.loc[df_ibge["REFE RÊNCIA"] == 2].assign(
    codigo_origem=lambda d: d["CÓDIGO DO ALIMENTO"],
    codigo_preparacao=lambda d: d["CÓDIGO DA PREPARAÇÃO"],
    nome=lambda d: d["DESCRIÇÃO NA REFERÊNCIA"].str.strip(),
)[["codigo_origem", "codigo_preparacao", "nome"]]

medidas_taco = (
    ligacao_taco.merge(df_medidas, on=["codigo_origem", "codigo_preparacao"])
    .groupby("nome")[COLUNAS_MEDIDAS]
    .median()
    .reset_index()
)

df_taco_final = df_taco_final.merge(medidas_taco, on="nome", how="left")

print("TACO com pelo menos 1 medida:", df_taco_final[COLUNAS_MEDIDAS].notna().any(axis=1).sum(), "de", len(df_taco_final))
df_taco_final[["nome"] + COLUNAS_MEDIDAS].dropna(subset=COLUNAS_MEDIDAS, how="all").head(10)

In [ ]:
#MEDIDAS NOS ALIMENTOS DO IBGE
# Ligação direta pelo código do alimento + código da preparação
df_ibge_final = df_ibge_final.merge(df_medidas, on=["codigo_origem", "codigo_preparacao"], how="left")

print("IBGE com pelo menos 1 medida:", df_ibge_final[COLUNAS_MEDIDAS].notna().any(axis=1).sum(), "de", len(df_ibge_final))
df_ibge_final[["nome"] + COLUNAS_MEDIDAS].dropna(subset=COLUNAS_MEDIDAS, how="all").head(10)

In [ ]:
#JUNTANDO TACO + IBGE
df_alimentos = pd.concat([df_taco_final, df_ibge_final], ignore_index=True)

print("Total de alimentos:", len(df_alimentos))
print(df_alimentos["fonte"].value_counts(), "\n")

print("Valores nulos por coluna:")
print(df_alimentos.groupby("fonte")[COLUNAS_ALIMENTOS].apply(lambda d: d.isna().sum()).T)

In [ ]:
#VALORES POR 1 GRAMA
# As tabelas TACO e IBGE trazem os nutrientes para 100 g do alimento.
# Dividindo por 100 temos o valor de 1 g, que serve de base para qualquer
# medida caseira: colher de chá, colher de sopa, xícara...
#   valor da porção = valor por 1 g x gramas da porção
# A umidade fica de fora porque é um percentual (%), não depende da quantidade.
COLUNAS_NUTRIENTES = [c for c in COLUNAS_ALIMENTOS if c not in ("nome", "umidade")]

for coluna in COLUNAS_NUTRIENTES:
    df_alimentos[f"{coluna}_1g"] = df_alimentos[coluna] / 100

df_alimentos[["nome", "energia_kcal", "energia_kcal_1g", "proteina", "proteina_1g"]].head()

In [ ]:
#EXEMPLO: CALCULANDO UMA PORÇÃO EM GRAMAS
# Os gramas de cada medida vêm dos modelos de dieta do grupo (backend/src/main/resources/data.sql):
#   "3 colheres de sopa cheias" de arroz integral cozido = 60 g
#   "2 colheres de sopa" de cenoura ralada crua          = 24 g
def calcular_porcao(nome, gramas, fonte="TACO"):
    alimento = df_alimentos[(df_alimentos["nome"] == nome) & (df_alimentos["fonte"] == fonte)].iloc[0]
    return pd.Series(
        {coluna: alimento[f"{coluna}_1g"] * gramas for coluna in COLUNAS_NUTRIENTES},
        name=f"{nome} ({gramas} g)"
    ).round(2)

pd.concat([
    calcular_porcao("Arroz, integral, cozido", 60),
    calcular_porcao("Cenoura, crua", 24),
], axis=1)

In [ ]:
#EXEMPLO: CALCULANDO UMA PORÇÃO EM MEDIDA CASEIRA
# Usa os gramas da Tabela de Medidas Referidas do IBGE:
#   gramas da porção = quantidade x gramas da medida
#   nutriente        = valor por 1 g x gramas da porção
def calcular_medida(nome, medida, quantidade, fonte="IBGE"):
    alimento = df_alimentos[(df_alimentos["nome"] == nome) & (df_alimentos["fonte"] == fonte)].iloc[0]
    gramas = alimento[f"gramas_{medida}"]
    if pd.isna(gramas):
        raise ValueError(f"{nome} não tem a medida {medida}")
    porcao = calcular_porcao(nome, quantidade * gramas, fonte)
    porcao.name = f"{nome} ({quantidade} {medida} = {quantidade * gramas:g} g)"
    return porcao

pd.concat([
    calcular_medida("Arroz integral", "colher_sopa", 3),
    calcular_medida("Ovo de galinha - Cozido(a)", "unidade", 2),
    calcular_medida("Leite de vaca integral", "xicara", 1),
    calcular_medida("Acucar", "colher_cha", 2),
], axis=1)

In [ ]:
#AMOSTRA DO RESULTADO
df_alimentos.sample(10, random_state=42)

In [ ]:
#EXPORTANDO CSV
# sep=";" e decimal="," para abrir direto no Excel em português
df_alimentos.to_csv(
    CAMINHO_SAIDA,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

print("Arquivo salvo:", CAMINHO_SAIDA)

In [ ]:
#GERANDO O SCRIPT SQL PARA O MYSQL
# Atenção ao padrão do banco: o data.sql do backend insere a TACO por 100 g e
# depois DIVIDE tudo por 100, ou seja, a tabela "alimentos" guarda os
# nutrientes POR 1 g. Por isso o IBGE entra com as colunas *_1g.
# As medidas caseiras ficam em gramas (não são divididas).
import os

CAMINHO_SQL = "../Query/Script medidas caseiras e alimentos IBGE.sql"

def valor_sql(valor):
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return "NULL"
    if isinstance(valor, str):
        return "'" + valor.replace("'", "''") + "'"
    return repr(round(float(valor), 6))

# 1) Uma coluna nova para cada medida do front
colunas_novas = {
    coluna: f"ALTER TABLE alimentos ADD COLUMN {coluna} DOUBLE NULL"
    for coluna in COLUNAS_MEDIDAS
}

comandos = []

# 2) TACO: os alimentos já existem no banco (data.sql) -> só preenche as medidas
taco_com_medida = df_alimentos[df_alimentos["fonte"] == "TACO"].dropna(subset=COLUNAS_MEDIDAS, how="all")
for _, linha in taco_com_medida.iterrows():
    sets = ", ".join(f"{c} = {valor_sql(linha[c])}" for c in COLUNAS_MEDIDAS)
    comandos.append(f"UPDATE alimentos SET {sets} WHERE fonte = 'TACO' AND nome = {valor_sql(linha['nome'])}")

# 3) IBGE: alimentos novos, valores por 1 g + medidas.
#    "WHERE NOT EXISTS" evita duplicar o alimento se rodar mais de uma vez
colunas_banco = {c: f"{c}_1g" for c in COLUNAS_NUTRIENTES}
colunas_banco["vitaminaC"] = colunas_banco.pop("vitaminac")  # nome usado no data.sql
colunas_insert = ["fonte", "nome"] + list(colunas_banco) + COLUNAS_MEDIDAS

# O backend busca alimento pelo nome exato (findByNome), então o nome não
# pode se repetir: quando o IBGE tem o mesmo nome da TACO, ganha " (IBGE)"
nomes_taco = set(df_alimentos.loc[df_alimentos["fonte"] == "TACO", "nome"])

for _, linha in df_alimentos[df_alimentos["fonte"] == "IBGE"].iterrows():
    nome = linha["nome"] + " (IBGE)" if linha["nome"] in nomes_taco else linha["nome"]
    valores = [valor_sql("IBGE"), valor_sql(nome)]
    valores += [valor_sql(linha[coluna_df]) for coluna_df in colunas_banco.values()]
    valores += [valor_sql(linha[c]) for c in COLUNAS_MEDIDAS]
    comandos.append(
        f"INSERT INTO alimentos ({', '.join(colunas_insert)}) "
        f"SELECT {', '.join(valores)} FROM DUAL "
        f"WHERE NOT EXISTS (SELECT 1 FROM alimentos WHERE fonte = 'IBGE' AND nome = {valor_sql(nome)})"
    )

# Salva o .sql (para rodar no MySQL Workbench uma vez só, por causa do ALTER TABLE)
os.makedirs(os.path.dirname(CAMINHO_SQL), exist_ok=True)  # o Git não guarda pasta vazia
with open(CAMINHO_SQL, "w", encoding="utf-8") as arquivo:
    arquivo.write("-- Gerado por dados/ETL/tratamento.py - não editar à mão\n")
    arquivo.write("-- Medidas caseiras (Tabela de Medidas Referidas, IBGE/POF 2008-2009)\n")
    arquivo.write("-- e alimentos do IBGE (valores por 1 g, mesmo padrão do data.sql)\n\n")
    arquivo.write("USE fittnutri;\n\n")
    arquivo.write("-- 1) Colunas das medidas do front (gramas de 1 unidade da medida)\n")
    arquivo.writelines(f"{comando};\n" for comando in colunas_novas.values())
    arquivo.write("\n-- 2) Medidas da TACO e 3) alimentos do IBGE\n")
    arquivo.writelines(f"{comando};\n" for comando in comandos)

print("Comandos:", len(colunas_novas), "ALTER,", len(taco_com_medida), "UPDATE,", len(comandos) - len(taco_com_medida), "INSERT")
print("Script SQL salvo:", CAMINHO_SQL)

In [ ]:
#CARREGANDO NO MYSQL
# Usa os dados da célula "#CONEXAO COM BD" (SOURCE_HOST, SOURCE_USER, SOURCE_PASSWORD)
# e o banco do backend (fittnutri). A tabela "alimentos" precisa existir:
# ela é criada quando o backend sobe pela primeira vez.
conexao = mysql.connector.connect(
    host=SOURCE_HOST,
    user=SOURCE_USER,
    password=SOURCE_PASSWORD,
    database="fittnutri"
)
cursor = conexao.cursor()

# ALTER TABLE só para as colunas que ainda não existem (permite rodar de novo)
cursor.execute(
    "SELECT COLUMN_NAME FROM information_schema.COLUMNS "
    "WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = 'alimentos'"
)
existentes = {nome.lower() for (nome,) in cursor.fetchall()}
assert existentes, "Tabela 'alimentos' não encontrada. Suba o backend uma vez para criá-la."

for coluna, comando in colunas_novas.items():
    if coluna not in existentes:
        cursor.execute(comando)  # ALTER TABLE faz commit sozinho no MySQL
        print("Coluna criada:", coluna)

# O data.sql usa "vitaminaC", mas o Hibernate pode ter criado "vitamina_c"
if "vitaminac" not in existentes and "vitamina_c" in existentes:
    comandos = [c.replace("vitaminaC", "vitamina_c") for c in comandos]

# UPDATE + INSERT numa transação: ou entra tudo, ou nada
try:
    atualizados = inseridos = 0
    for comando in comandos:
        cursor.execute(comando)
        if comando.startswith("UPDATE"):
            atualizados += cursor.rowcount
        else:
            inseridos += cursor.rowcount
    conexao.commit()
except Exception:
    conexao.rollback()
    raise
finally:
    cursor.close()
    conexao.close()

print(f"MySQL: {atualizados} alimentos da TACO com medidas, {inseridos} alimentos do IBGE inseridos")